# Limpieza de datos proyecto Gaia Estrellas
Comenzamos cargando las importaciones que seran necesarias en la realización de este proceso de los datos referentes al archivo de la "Gaia Stars" que hemos ya preseleccionado para tener un tamaño de datos acorde para este proyecto sin ser este excesibamente masibo.

In [ ]:
import pandas as pd
from pathlib import Path

Se crean las variables que nos daran aceso a cargar y guardar nuestros archivos .cvs

In [ ]:
# Configuraciones de rutas (pathlib resuelve conflictos rutas linux y windows)
BASE_PATH = Path().resolve()
DATA_RAW = BASE_PATH / ".." / "data" / "raw"
DATA_PROCESSED = BASE_PATH / ".." / "data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

file_path = DATA_RAW / "neighbor_stars.csv"

In [15]:
# Lectura del arcchivo csv raw
df = pd.read_csv(file_path)

# Informacion del dataframe
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

# Impresion de primeras filas
df.head()

Filas: 102625
Columnas: 7


,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692


In [16]:
print("Filas:", len(df))
print("Estrellas:", df["source_id"].nunique())

Filas: 102625
Estrellas: 102625


In [17]:
# Informacion basica del dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102625 entries, 0 to 102624
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   source_id            102625 non-null  int64  
 1   ra                   102625 non-null  float64
 2   dec                  102625 non-null  float64
 3   parallax             102625 non-null  float64
 4   parallax_over_error  102625 non-null  float64
 5   phot_g_mean_mag      102439 non-null  float64
 6   bp_rp                92863 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 5.5 MB


Aqui se exploraran datos muy valiosos sobre el dataframe, desde datos minimos de las diferentes variables a datos medios por porcentajes.

In [18]:
# Descropcion tecnica del dataframe
df.describe()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp
count,1.026250e+05,102625.000000,102625.000000,102625.000000,102625.000000,102439.000000,92863.000000
mean,3.701500e+18,203.316305,-5.927630,23.415270,596.077739,15.506059,2.302688
std,1.852270e+18,97.404131,36.783259,12.124055,724.814247,4.029278,0.952244
min,1.234245e+14,0.005121,-89.461367,16.400141,5.000787,2.016425,-1.380596
25%,2.077398e+18,121.519653,-32.515420,17.837748,15.036133,12.863765,1.624001
50%,4.059441e+18,236.874676,-11.731849,20.072150,334.594730,15.342374,2.380388
75%,5.210778e+18,274.576872,21.907549,24.534712,992.499760,19.659054,2.923058
max,6.917452e+18,359.994771,89.581055,768.066539,15400.477000,21.289928,5.913773


Comenzamos contamos los valores nulos y los valores duplicados.

Terminamos eliminando los valores duplicados de nuestro dataframe, seleccionados minuciosamente por nombre de planeta, ya que la funcion de eliminacion de duplicados discrimina y elimina muchos registros por ser muy parecidos.

In [19]:
print("\nValores nulos por columna:\n")
print(df.isnull().sum())

print("\nTotal de valores nulos:", df.isnull().sum().sum())

print("\nFilas duplicadas:", df.duplicated().sum())

print("Filas antes:", len(df))

df = df.drop_duplicates(subset=["source_id"])

print("Filas después:", len(df))


Valores nulos por columna:

source_id                 0
ra                        0
dec                       0
parallax                  0
parallax_over_error       0
phot_g_mean_mag         186
bp_rp                  9762
dtype: int64

Total de valores nulos: 9948

Filas duplicadas: 0
Filas antes: 102625
Filas después: 102625


# Revisión de valores

Se revisan los valores referentes de la posicion espacial ra, dec y la distancia con la tierra para explorar la necesidad de eliminar valores fuera de rango, outliners o posibles errores sin sentido como una dec de mas de 90 grados saluendose del plano visual.

In [20]:
print("Inicial:", len(df))

# RA
df = df[(df["ra"] >= 0) & (df["ra"] <= 360)]
print("Tras filtro RA:", len(df))

# DEC
df = df[(df["dec"] >= -90) & (df["dec"] <= 90)]
print("Tras filtro DEC:", len(df))

# Distancia
df = df[df["parallax"] > 0]
print("Tras filtro distancia:", len(df))

Inicial: 102625
Tras filtro RA: 102625
Tras filtro DEC: 102625
Tras filtro distancia: 102625


### Aplico filto de error en medicion de distancia
Con tal de verificar con mayor seguridad se aplica un filtro de calidad de la medicion de distancia en el parallax, para asegurarnos posibles errores en esta variable

In [21]:
print("Antes de filtro de errores de distancia:", len(df))

df = df[df["parallax_over_error"] > 5]

print("Tras filtro distancia:", len(df))

Antes de filtro de errores de distancia: 102625
Tras filtro distancia: 102625


Antes de proceder a guardar todo el dataframe como un archivo cvs procesado, se va a añadir una fila nueva llamada distance_pc que coincida el valor de la nasa con el de gaia con tal de simplificar los match que realizaremos despues con ambas tablas. Tambien se revisa cuantos sistemas diferentes se encuentran, esta cifra deve coincidir en numero

In [22]:
df["distance_pc"] = 1000 / df["parallax"]

df.head()

,source_id,ra,dec,parallax,parallax_over_error,phot_g_mean_mag,bp_rp,distance_pc
0,1495713771471104,42.344198,2.290799,18.096821,823.04285,12.842120,2.263671,55.258325
1,2183668748454272,44.753144,3.972805,18.404966,697.28980,14.311949,2.499146,54.333162
2,3656185400409472,46.965442,5.236058,19.264496,388.73822,15.862782,3.269577,51.908963
3,6347480627407104,42.935498,5.651869,17.650109,1014.52515,11.135140,1.621806,56.656875
4,6397856299125120,43.343943,6.132225,22.178355,451.88208,15.695479,3.317692,45.089007


In [23]:
print("Numero de registros:", len(df))
print("Número de sistemas:", df["source_id"].nunique())

Numero de registros: 102625
Número de sistemas: 102625


In [ ]:
# Guardar final
output_path = DATA_PROCESSED / "gaia_neighbor_processed.csv"

df.to_csv(output_path, index=False)

print(f"Archivo Gaia guardado en: {output_path}")
print(f"Resumen: {len(df)} estrellas procesadas.")

Guardado en: F:\Usuario\Diego\Estudios\Ilerna\EspacializacionBigDataIA\ProyectoBigData\proyectoBCSS\notebook\..\data\processed\gaia_neighbor_processed.csv
